In [0]:
import logging
from itertools import chain
from pyspark.sql import functions as F

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("SilverVeiculosEletricos")

# Mapeamento oficial Nome por Extenso -> Sigla UF (para padronizar a base da SENATRAN)
mapa_estados_uf = {
    "ACRE": "AC", "ALAGOAS": "AL", "AMAPA": "AP", "AMAPÁ": "AP",
    "AMAZONAS": "AM", "BAHIA": "BA", "CEARA": "CE", "CEARÁ": "CE",
    "DISTRITO FEDERAL": "DF", "ESPIRITO SANTO": "ES", "ESPÍRITO SANTO": "ES",
    "GOIAS": "GO", "GOIÁS": "GO", "MARANHAO": "MA", "MARANHÃO": "MA",
    "MATO GROSSO": "MT", "MATO GROSSO DO SUL": "MS", "MINAS GERAIS": "MG",
    "PARA": "PA", "PARÁ": "PA", "PARAIBA": "PB", "PARAÍBA": "PB",
    "PARANA": "PR", "PARANÁ": "PR", "PERNAMBUCO": "PE", "PIAUI": "PI",
    "PIAUÍ": "PI", "RIO DE JANEIRO": "RJ", "RIO GRANDE DO NORTE": "RN",
    "RIO GRANDE DO SUL": "RS", "RONDONIA": "RO", "RONDÔNIA": "RO",
    "RORAIMA": "RR", "SANTA CATARINA": "SC", "SAO PAULO": "SP",
    "SÃO PAULO": "SP", "SERGIPE": "SE", "TOCANTINS": "TO"
}

mapping_expr_uf = F.create_map([F.lit(x) for x in chain(*mapa_estados_uf.items())])

In [0]:
logger.info("Processando silver_emplacamentos_veiculos...")

df_silver_emp = (
    spark.table("bronze_emplacamentos_veiculos")
    .filter(F.col("uf").isNotNull() & (F.col("uf") != "BR"))
    .withColumn("ano", F.split(F.col("ano_mes"), "-").getItem(0).cast("int"))
    .withColumn("mes_num", F.split(F.col("ano_mes"), "-").getItem(1).cast("int"))
    .withColumn(
        "grupo_tecnologia",
        F.when(F.col("tecnologia").isin("BEV", "PHEV"), F.lit("Plug-in (BEV/PHEV)"))
        .otherwise(F.lit("Híbrido Convencional (HEV/MHEV)"))
    )
    .withColumn("exige_recarga_externa", F.col("tecnologia").isin("BEV", "PHEV"))
    .select(
        "ano_mes", "ano", "mes_num", "uf", "posicao_ranking",
        "marca", "modelo", "tecnologia", "grupo_tecnologia",
        "exige_recarga_externa", "qtd_emplacamentos"
    )
)

df_silver_emp.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_emplacamentos_veiculos")
logger.info(f"silver_emplacamentos_veiculos gravada: {df_silver_emp.count()} registros.")

In [0]:
logger.info("Processando silver_frota_senatran...")

combustiveis_plugin = [
    "ELETRICO/FONTE EXTERNA",
    "HIBRIDO PLUG-IN",
    "ELETRICO",
    "CELULA COMBUSTIVEL"
]

combustiveis_hibridos = [
    "GASOLINA/ELETRICO",
    "GASOLINA/ALCOOL/ELETRICO",
    "ELETRICO/FONTE INTERNA",
    "HIBRIDO",
    "DIESEL/ELETRICO",
    "ETANOL/ELETRICO",
    "HIBRIDO/GAS NATURAL VEICULAR"
]

todos_eletrificados = combustiveis_plugin + combustiveis_hibridos

df_silver_senatran = (
    spark.table("bronze_frota_senatran")
    .filter(F.col("tipo_combustivel").isin(todos_eletrificados))
    .withColumn("uf", F.coalesce(mapping_expr_uf[F.trim(F.col("estado_nome"))], F.lit("DESCONHECIDO")))
    .withColumn(
        "categoria_frota",
        F.when(F.col("tipo_combustivel").isin(combustiveis_plugin), F.lit("Plug-in (Elétrico / PHEV)"))
        .otherwise(F.lit("Híbrido sem recarga externa"))
    )
    .withColumn("exige_recarga_externa", F.col("tipo_combustivel").isin(combustiveis_plugin))
    .select(
        "uf",
        F.col("estado_nome").alias("estado_extenso"),
        F.trim(F.col("municipio")).alias("municipio"),
        "tipo_combustivel",
        "categoria_frota",
        "exige_recarga_externa",
        "qtd_veiculos"
    )
)

df_silver_senatran.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_frota_senatran")
logger.info(f"silver_frota_senatran gravada: {df_silver_senatran.count()} registros.")

In [0]:
logger.info("Processando silver_ipva_veiculos_2026...")

df_silver_ipva = (
    spark.table("bronze_ipva_veiculos_2026")
    .withColumn(
        "aliquota_eletrico_efetiva",
        F.when(F.col("regra_ipva_eletrico") == "isento", F.lit(0.0))
        .when(F.col("aliquota_eletrico_pct").isNotNull(), F.col("aliquota_eletrico_pct"))
        .otherwise(F.col("aliquota_geral_pct"))
    )
    .withColumn(
        "incentivo_fiscal_score",
        F.when(F.col("regra_ipva_eletrico") == "isento", F.lit("Alto (Isenção Total)"))
        .when(F.col("regra_ipva_eletrico") == "reduzido", F.lit("Médio (Alíquota Reduzida)"))
        .otherwise(F.lit("Baixo (Alíquota Integral)"))
    )
    .select(
        "uf", "nome_estado", "regra_ipva_eletrico", "aliquota_eletrico_efetiva",
        "regra_ipva_hibrido", "aliquota_geral_pct", "incentivo_fiscal_score", "resumo_ipva_eletrico"
    )
)

df_silver_ipva.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_ipva_veiculos_2026")

logger.info("Processando silver_ficha_tecnica_veiculos...")

df_silver_ficha = (
    spark.table("bronze_ficha_tecnica_veiculos")
    .filter(F.col("condicao") == "novo")
    .withColumn(
        "faixa_preco",
        F.when(F.col("preco_referencia_brl") <= 150000, F.lit("Entrada (até R$ 150k)"))
        .when(F.col("preco_referencia_brl") <= 250000, F.lit("Intermediário (R$ 150k - 250k)"))
        .when(F.col("preco_referencia_brl") <= 400000, F.lit("Premium (R$ 250k - 400k)"))
        .otherwise(F.lit("Luxo (acima de R$ 400k)"))
    )
)

df_silver_ficha.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_ficha_tecnica_veiculos")
logger.info("Camada SILVER 100% concluída!")